# Interactive simulation checks: effect propagation

Compares baseline vs the `mms_total_scaleup` scenario (common random numbers) to verify how
the oral-iron intervention propagates: it shifts the *pipeline* hemoglobin and gestational-age
birth exposure, but not the *state-table* values or the maternal-hemorrhage risk, and it moves
the preterm relative risks as expected. Ported from the research portfolio VnV notebook
`model_18.3_interactive_simulation_effect_propogation`; updated to the current Engine
(`vivarium.engine`) API and converted to assert-based checks.

Note: the source's exploratory stillbirth-ratio cells were left out (the finding there was
ambiguous); add targeted stillbirth checks once that behavior is confirmed.

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

import vivarium_gates_mncnh
from vivarium.artifact import Artifact
from vivarium.engine import InteractiveContext
from vivarium.engine.framework.configuration import build_model_specification

In [ ]:
!pip list | grep vivarium

In [ ]:
SPEC_PATH = Path(vivarium_gates_mncnh.__file__).parent / "model_specifications/model_spec.yaml"
COLS = ["anc_attendance", "oral_iron_intervention", "age", "hemoglobin_exposure",
        "maternal_hemorrhage", "pregnancy_outcome", "gestational_age_exposure"]
PIPELINES = ["maternal_hemorrhage.incidence_risk", "maternal_hemorrhage.incidence_risk.paf",
             "hemoglobin_on_maternal_hemorrhage.relative_risk", "hemoglobin.exposure",
             "gestational_age.birth_exposure"]

def run_to_hemorrhage(scenario=None):
    spec = build_model_specification(SPEC_PATH)
    del spec.configuration.observers
    spec.configuration.population.population_size = 20_000 * 10
    if scenario is not None:
        spec.configuration.intervention.scenario = scenario
    sim = InteractiveContext(spec)
    get_event_name = sim._builder.time.simulation_event_name()
    while get_event_name() != "maternal_hemorrhage":
        sim.step()
    sim.step()  # advance past maternal_hemorrhage
    return sim

def frame(sim):
    pop = sim.get_population()
    vals = pd.concat([sim.get_value(p)(pop.index).rename(p) for p in PIPELINES], axis=1)
    return pd.concat([pop[COLS], vals], axis=1)

In [ ]:
base_spec = build_model_specification(SPEC_PATH)
art = Artifact(base_spec.configuration.input_data.artifact_path)
draw = "draw_" + str(base_spec.configuration.input_data.input_draw_number)

baseline = run_to_hemorrhage()
mms = run_to_hemorrhage("mms_total_scaleup")
comp = frame(baseline).merge(frame(mms), left_index=True, right_index=True, suffixes=["_baseline", "_mms"])
comp.head()

## MMS does not change the state table or the hemorrhage risk

In [ ]:
# State-table hemoglobin, the maternal-hemorrhage risk/PAF/RR, and (with common random
# numbers) the hemorrhage outcome are all identical across scenarios -- hemorrhage risk is
# driven by the state-table exposure, which MMS does not modify.
for col in ["hemoglobin_exposure", "maternal_hemorrhage.incidence_risk",
            "maternal_hemorrhage.incidence_risk.paf", "hemoglobin_on_maternal_hemorrhage.relative_risk"]:
    assert np.allclose(comp[f"{col}_baseline"], comp[f"{col}_mms"]), \
        f"{col} changed between scenarios (expected identical)"
assert (comp["maternal_hemorrhage_baseline"] == comp["maternal_hemorrhage_mms"]).all(), \
    "maternal_hemorrhage outcome changed between scenarios (common random numbers should match)"

## MMS shifts the pipeline hemoglobin and gestational age

In [ ]:
# Pipeline hemoglobin and GA birth-exposure DO rise under MMS, and the pipeline hemoglobin
# differs from the state-table value.
assert comp["hemoglobin.exposure_mms"].mean() > comp["hemoglobin.exposure_baseline"].mean(), \
    "MMS did not raise pipeline hemoglobin"
assert not np.allclose(comp["hemoglobin_exposure_baseline"], comp["hemoglobin.exposure_baseline"]), \
    "state-table and pipeline hemoglobin are unexpectedly identical"
assert np.allclose(comp["gestational_age_exposure_baseline"], comp["gestational_age_exposure_mms"]), \
    "state-table gestational age changed between scenarios"
assert comp["gestational_age.birth_exposure_mms"].mean() > comp["gestational_age.birth_exposure_baseline"].mean(), \
    "MMS did not raise pipeline GA birth exposure"

In [ ]:
# For simulants switching no_treatment -> MMS, the observed GA shift matches the artifact's
# MMS + IFA excess shifts (observed is expected to run a touch higher, since a larger shift
# applies to the GA<32wk minority), so use a generous tolerance.
mms_ga_shift = art.load("risk_factor.multiple_micronutrient_supplementation.excess_gestational_age_shift_subpop_2")[draw].values[1]
ifa_ga_shift = art.load("risk_factor.iron_folic_acid_supplementation.excess_shift")[draw].values[-1]
expected_shift = mms_ga_shift + ifa_ga_shift
switchers = comp[(comp.oral_iron_intervention_baseline == "no_treatment") & (comp.oral_iron_intervention_mms == "mms")]
observed_shift = (switchers["gestational_age.birth_exposure_mms"] - switchers["gestational_age.birth_exposure_baseline"]).mean()
assert observed_shift >= expected_shift * 0.9, f"observed GA shift {observed_shift:.3f} well below expected ~{expected_shift:.3f}"
assert abs(observed_shift - expected_shift) < 0.2, f"observed GA shift {observed_shift:.3f} far from expected {expected_shift:.3f}"

## Preterm relative risks are protective

In [ ]:
# IFA (vs no treatment) and MMS (vs IFA) each reduce preterm birth (RR < 1, ~0.9).
comp["preterm_baseline"] = comp["gestational_age.birth_exposure_baseline"] < 37
comp["preterm_mms"] = comp["gestational_age.birth_exposure_mms"] < 37

none_mask = (comp.oral_iron_intervention_baseline == "no_treatment") & (comp.anc_attendance_baseline != "none")
preterm_none = comp.loc[none_mask, "preterm_baseline"].mean()
preterm_ifa = comp.loc[comp.oral_iron_intervention_baseline == "ifa", "preterm_baseline"].mean()
preterm_mms = comp.loc[comp.oral_iron_intervention_mms == "mms", "preterm_mms"].mean()
rr_ifa = preterm_ifa / preterm_none
rr_mms = preterm_mms / preterm_ifa
assert 0.80 < rr_ifa < 1.0, f"IFA preterm RR {rr_ifa:.3f} not protective / outside ~0.9"
assert 0.80 < rr_mms < 1.0, f"MMS-vs-IFA preterm RR {rr_mms:.3f} not protective / outside ~0.9"